In [ ]:
# Cell 1: 基础环境 & 随机种子

import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, recall_score

import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

def set_seed(seed=51):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(51)

# 加载 hetero_data.pt，并准备元信息

HET_PATH = os.path.join(".", "input_data", "hetero_data.pt")
data_obj = torch.load(HET_PATH)

if isinstance(data_obj, HeteroData):
    hetero_data = data_obj
elif isinstance(data_obj, dict) and 'hetero_data' in data_obj:
    hetero_data = data_obj['hetero_data']
else:
    raise ValueError("无法识别 hetero_data 格式，请确认 hetero_data.pt 里存的是 HeteroData 或 {'hetero_data': HeteroData}.")

print(hetero_data)

# 元信息
node_types, edge_types = hetero_data.metadata()
print("Node types:", node_types)
print("Edge types:", edge_types)

# 每种节点类型的输入维度
in_dims = {ntype: hetero_data[ntype].x.size(1) for ntype in node_types}
print("Input dims per node type:", in_dims)

# 拷贝一份特征到 CPU（训练时再整体搬到 device）
x_dict = {ntype: hetero_data[ntype].x.clone() for ntype in node_types}


Device: cuda
HeteroData(
  protein={
    x=[18326, 150],
    y=[18326],
  },
  lncrna={ x=[1811, 150] },
  mirna={ x=[1589, 150] },
  (lncrna, lncrnaTOlncrna, lncrna)={ edge_index=[2, 296] },
  (lncrna, lncrnaTOmirna, mirna)={ edge_index=[2, 7277] },
  (lncrna, lncrnaTOprotein, protein)={ edge_index=[2, 10376] },
  (mirna, mirnaTOlncrna, lncrna)={ edge_index=[2, 7277] },
  (mirna, mirnaTOmirna, mirna)={ edge_index=[2, 158] },
  (mirna, mirnaTOprotein, protein)={ edge_index=[2, 510229] },
  (protein, proteinTOlncrna, lncrna)={ edge_index=[2, 10376] },
  (protein, proteinTOmirna, mirna)={ edge_index=[2, 510229] },
  (protein, proteinTOprotein, protein)={ edge_index=[2, 939078] }
)
Node types: ['protein', 'lncrna', 'mirna']
Edge types: [('lncrna', 'lncrnaTOlncrna', 'lncrna'), ('lncrna', 'lncrnaTOmirna', 'mirna'), ('lncrna', 'lncrnaTOprotein', 'protein'), ('mirna', 'mirnaTOlncrna', 'lncrna'), ('mirna', 'mirnaTOmirna', 'mirna'), ('mirna', 'mirnaTOprotein', 'protein'), ('protein', 'proteinTO

In [28]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, recall_score

# ===============================
# 0) 固定 seed
# ===============================
SEED = 51
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 可复现（会稍慢）
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ===============================
# 1) 检查标签
# ===============================
if 'y' not in hetero_data['protein']:
    raise ValueError("protein 节点缺少标签 y，请在 hetero_data['protein'].y 中提供标签（0/1，未标注可为-1）。")

protein_y = hetero_data['protein'].y.view(-1)          # CPU
labeled_mask = protein_y >= 0
labeled_idx = labeled_mask.nonzero(as_tuple=False).view(-1)
labeled_y = protein_y[labeled_idx].cpu().numpy()

print(f"Total protein nodes     : {hetero_data['protein'].num_nodes}")
print(f"Labeled protein nodes   : {labeled_idx.numel()}")

Total protein nodes     : 18326
Labeled protein nodes   : 2856


In [ ]:
with open('./input_data/all_gene_names.json', 'r') as f:
    name = json.load(f)

protein_names = name['protein']

In [45]:
import torch
import torch.nn.functional as F
import numpy as np
import os, json,csv
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc

# 评估指标
def evaluate_metrics(probs, true):
    pred = np.argmax(probs, axis=1)
    acc = accuracy_score(true, pred)
    recall = recall_score(true, pred, average='macro')
    f1 = f1_score(true, pred, average='macro')
    auroc = roc_auc_score(true, probs[:, 1])
    prec, rec, _ = precision_recall_curve(true, probs[:, 1])
    auprc = auc(rec, prec)
    return {
        'accuracy': acc,
        'recall': recall,
        'f1': f1,
        'auroc': auroc,
        'auprc': auprc
    }
    
def save_all_probs_csv(protein_names, probs, labels, filename):
    """
    protein_names: list[str]，长度等于所有protein节点数量
    probs: np.array, shape=(N,)，所有protein的概率
    labels: np.array, shape=(N,)，所有protein的标签（或test掩码下的标签）
    filename: str
    """
    # 检查长度一致
    assert len(protein_names) == len(probs) == len(labels)
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['protein', 'prob', 'label'])
        for name, prob, label in zip(protein_names, probs, labels):
            writer.writerow([name, prob, label])
            
# 分类器
class NodeClassifier(torch.nn.Module):
    def __init__(self, embed_dim, num_classes):
        super(NodeClassifier, self).__init__()
        self.fc1 = nn.Linear(embed_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [31]:
labels = hetero_data['protein'].y
labels_np = labels[labeled_idx].cpu().numpy()
mask_idx = protein_y >= 0

In [33]:
sum(mask_idx)

tensor(2856)

In [46]:
def train_model_with_cv(path,model_name, hetero_data, model_builder, device='cuda' if torch.cuda.is_available() else 'cpu'):
    os.makedirs(f"{path}/cv_results_{model_name}", exist_ok=True)

    #rgcn全图表示
    x_all, edge_index_all, edge_type_all, node_offset = extract_rgcn_data(hetero_data)
    x_all = x_all.to(device)
    edge_index_all = edge_index_all.to(device)
    edge_type_all = edge_type_all.to(device)
    val_results, test_results = [], []
    hetero_data = hetero_data.to(device)
        
    labels = hetero_data['protein'].y.view(-1)
    
    # 只保留有标签节点
    labeled_mask = labels >= 0
    mask_idx = labeled_mask.nonzero(as_tuple=True)[0]
    
    labels_np = labels[mask_idx].cpu().numpy()
    
    train_idx, test_idx = train_test_split(
        mask_idx.cpu().numpy(),
        test_size=0.25,
        stratify=labels_np,
        random_state=51
    )       
    
    with open(f"{path}/cv_results_{model_name}/train_test_split.json", 'w') as f:
        json.dump({'train_idx': train_idx.tolist(), 'test_idx': test_idx.tolist()}, f, indent=4)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=51)


    for fold, (sub_train, val) in enumerate(skf.split(train_idx, labels[train_idx].cpu().numpy())):
        print(f"\n[{model_name}] Fold {fold+1}/10")
        fold_dir = f"{path}/cv_results_{model_name}/fold_{fold+1}"
        os.makedirs(fold_dir, exist_ok=True)

        sub_train = torch.tensor(train_idx[sub_train], device=device)
        val = torch.tensor(train_idx[val], device=device)
        test = torch.tensor(test_idx, device=device)

        model = model_builder().to(device)
        classifier = NodeClassifier(embed_dim=90, num_classes=2).to(device)
        optimizer = torch.optim.Adam(list(model.parameters()) + list(classifier.parameters()), lr=1e-3)

        best_auprc = 0.0
        best_state = {}
        # === 损失函数 ===
        loss_fn = torch.nn.CrossEntropyLoss()

        for epoch in range(300):
            model.train(); classifier.train()
            if model_name == "rgcn":
                out_dict = model(x_all, edge_index_all, edge_type_all)
            elif model_name == "hgt":
                out_dict = model(hetero_data.x_dict, hetero_data.edge_index_dict)
            elif model_name == "han":
                out_dict = model(hetero_data['protein'].x)
            else: 
                raise ValueError(f"unknown model name:{model_name}")
            logits = classifier(out_dict['protein'])
            loss = loss_fn(logits[sub_train], labels.to(device)[sub_train])
            optimizer.zero_grad(); loss.backward(); optimizer.step()

            model.eval(); classifier.eval()
            with torch.no_grad():
                if model_name == "rgcn":
                    out_dict = model(x_all, edge_index_all, edge_type_all)
                elif model_name == "hgt":
                    out_dict = model(hetero_data.x_dict, hetero_data.edge_index_dict)
                elif model_name == "han":
                    out_dict = model(hetero_data['protein'].x)
                val_logits = classifier(out_dict['protein'])[val]
                val_probs = val_logits.softmax(dim=1).cpu().numpy()
                val_true = labels[val].cpu().numpy()
                result = evaluate_metrics(val_probs, val_true)
                if result['auprc'] > best_auprc:
                    best_auprc = result['auprc']
                    best_state = {
                        'epoch': epoch,
                        'model': model.state_dict(),
                        'classifier': classifier.state_dict(),
                        'metrics': result
                    }

        # 保存训练结果
        torch.save(best_state, f"{fold_dir}/best_model.pt")
        with open(f"{fold_dir}/metrics.json", 'w') as f:
            json.dump({'epoch': best_state['epoch'], **best_state['metrics']}, f, indent=4)
        val_results.append(best_state['metrics'])

        # 测试集
        model.load_state_dict(best_state['model'])
        classifier.load_state_dict(best_state['classifier'])
        model.eval(); classifier.eval()
        with torch.no_grad():
            if model_name == "rgcn":
                out_dict = model(x_all, edge_index_all, edge_type_all)
            elif model_name == "hgt":
                out_dict = model(hetero_data.x_dict, hetero_data.edge_index_dict)
            elif model_name == "han":
                out_dict = model(hetero_data['protein'].x)
            logits = classifier(out_dict['protein'])
            logits_test = logits[test]
            probs = logits.softmax(dim=1).cpu().numpy()
            probs_test = probs[test.cpu()]
            cancer_probs = probs[:,1]
            true = labels[test].cpu().numpy()
            result = evaluate_metrics(probs_test, true)
            with open(f"{fold_dir}/test_metrics.json", 'w') as f:
                json.dump(result, f, indent=4)

            labels_cpu = labels.cpu().numpy()
            save_all_probs_csv(protein_names, cancer_probs, labels_cpu, f"{fold_dir}/test_probs.csv")
            test_results.append(result)

    with open(f"{path}/cv_results_{model_name}/all_folds_metrics.json", 'w') as f:
        json.dump(val_results, f, indent=4)
    with open(f"{path}/cv_results_{model_name}/final_test_average.json", 'w') as f:
        avg = {k: float(np.mean([r[k] for r in test_results])) for k in test_results[0]}
        json.dump(avg, f, indent=4)
    print(f"{model_name} 完成训练，平均测试结果：", avg)
    


In [47]:
import torch
import torch.nn as nn
from torch_geometric.nn import HGTConv

class HGTModel(nn.Module):
    def __init__(self, metadata, in_channels, hidden_channels, out_channels, num_layers=2, heads=2):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(HGTConv(
                in_channels if i == 0 else hidden_channels,
                hidden_channels,
                metadata,
                heads=heads
            ))
        self.out_proj = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        for layer in self.layers:
            x_dict = layer(x_dict, edge_index_dict)
        return {"protein": self.out_proj(x_dict['protein'])}

In [48]:
train_model_with_cv("result/","hgt", hetero_data, lambda: HGTModel(
    metadata=hetero_data.metadata(),
    in_channels=150,
    hidden_channels=90,
    out_channels=90,
    num_layers=2,
    heads=2
))



[hgt] Fold 1/10

[hgt] Fold 2/10

[hgt] Fold 3/10

[hgt] Fold 4/10

[hgt] Fold 5/10

[hgt] Fold 6/10

[hgt] Fold 7/10

[hgt] Fold 8/10

[hgt] Fold 9/10

[hgt] Fold 10/10
hgt 完成训练，平均测试结果： {'accuracy': 0.7221288515406163, 'recall': 0.6629677267327121, 'f1': 0.6592061832554855, 'auroc': 0.6968212746320533, 'auprc': 0.5223711994682447}


In [49]:
import torch
from torch_geometric.nn import RGCNConv
import torch.nn as nn

class RGCNModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)

    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = torch.relu(x)
        x = self.conv2(x, edge_index, edge_type)
        return x



In [50]:
# Extract data from hetero HeteroData for RGCN: multi-node-type, multi-relation setting
def extract_rgcn_data(hetero_data):
    x_dict = hetero_data.x_dict
    x_all = torch.cat([x for x in x_dict.values()], dim=0)

    node_offset = {}
    offset = 0
    for ntype, x in x_dict.items():
        node_offset[ntype] = offset
        offset += x.size(0)

    edge_index_list = []
    edge_type_list = []
    rel_map = {}
    rel_id = 0
    for rel in hetero_data.edge_index_dict:
        if rel not in rel_map:
            rel_map[rel] = rel_id
            rel_id += 1
        src, dst = hetero_data[rel].edge_index
        src_offset = node_offset[rel[0]]
        dst_offset = node_offset[rel[2]]
        edge_index_list.append(torch.stack([src + src_offset, dst + dst_offset], dim=0))
        edge_type_list.append(torch.full((src.size(0),), rel_map[rel], dtype=torch.long))

    edge_index_all = torch.cat(edge_index_list, dim=1)
    edge_type_all = torch.cat(edge_type_list, dim=0)
    return x_all, edge_index_all, edge_type_all, node_offset

In [51]:
# 封装rgcn 便于train_model_with_cv
class RGCNWrapper(nn.Module):
    def __init__(self, model, node_offset, out_dim):
        super().__init__()
        self.model = model
        self.node_offset = node_offset
        self.out_proj = nn.Linear(out_dim, out_dim)

    def forward(self, x_all, edge_index_all, edge_type_all):
        out_all = self.model(x_all, edge_index_all, edge_type_all)
        protein_start = self.node_offset['protein']
        protein_end = self.node_offset['protein'] + hetero_data['protein'].x.size(0)
        out_dict = {"protein": self.out_proj(out_all[protein_start:protein_end])}
        return out_dict

In [52]:

x_all, edge_index_all, edge_type_all, node_offset = extract_rgcn_data(hetero_data)
num_relations = int(edge_type_all.max().item()) + 1

train_model_with_cv(
    "result",
    "rgcn",
    hetero_data,
    model_builder=lambda: RGCNWrapper(
        RGCNModel(
            in_channels=150,
            hidden_channels=90,
            out_channels=90,
            num_relations=num_relations
        ),
        node_offset=node_offset,
        out_dim=90
    ),
    device='cuda'
)


[rgcn] Fold 1/10

[rgcn] Fold 2/10

[rgcn] Fold 3/10

[rgcn] Fold 4/10

[rgcn] Fold 5/10

[rgcn] Fold 6/10

[rgcn] Fold 7/10

[rgcn] Fold 8/10

[rgcn] Fold 9/10

[rgcn] Fold 10/10
rgcn 完成训练，平均测试结果： {'accuracy': 0.7222689075630253, 'recall': 0.6506539092185492, 'f1': 0.6510144051748843, 'auroc': 0.6961595106481163, 'auprc': 0.4603333792704432}


In [53]:
metapaths = [
    ['protein', 'protein', 'protein'],          # PPP
    ['protein', 'mirna', 'protein'],            # PMP
    ['protein', 'lncrna', 'protein']            # PLP
]

from torch_geometric.data import Data
from torch_sparse import SparseTensor

def build_sparse(edge_index, num_nodes, device='cpu'):
    return SparseTensor(row=edge_index[0], col=edge_index[1], sparse_sizes=(num_nodes, num_nodes)).to(device)

def extract_metapath_graphs_fast(hetero_data, metapaths, max_edges_per_path=400_000):
    metapath_graphs = {}
    for path in metapaths:
        try:
            # 使用 CPU 执行稀疏矩阵构造与乘法，防止爆显存
            num_nodes = hetero_data['protein'].x.size(0)
            adj = None
            for i in range(len(path) - 1):
                src, dst = path[i], path[i+1]
                key = next(k for k in hetero_data.edge_index_dict if k[0] == src and k[2] == dst)
                edge_index = hetero_data[key].edge_index.cpu()
                size = hetero_data[src].x.size(0), hetero_data[dst].x.size(0)
                sparse = build_sparse(edge_index, max(size), device='cpu')
                adj = sparse if adj is None else adj @ sparse

            row, col, _ = adj.coo()
            edge_index = torch.stack([row, col], dim=0)

            # 控制边数（仍在 CPU 上）
            num_edges = edge_index.size(1)
            if num_edges > max_edges_per_path:
                perm = torch.randperm(num_edges)[:max_edges_per_path]
                edge_index = edge_index[:, perm]

            # 构建 hetero Data 对象
            x = hetero_data['protein'].x.cpu()  # 先留在 CPU，后续可 .to(device)
            metapath_graphs['TO'.join(path)] = Data(x=x, edge_index=edge_index)

            print(f"Path {'TO'.join(path)}: {edge_index.size(1):,} edges.")

        except StopIteration:
            print(f"Meta-path {path} skipped due to missing edge.")
    return metapath_graphs
metapath_graphs = extract_metapath_graphs_fast(hetero_data, metapaths, max_edges_per_path=1_200_000)


/root/miniconda3/lib/python3.12/site-packages/torch_sparse/matmul.py:97: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  C = torch.sparse.mm(A, B)


Path proteinTOproteinTOprotein: 1,200,000 edges.
Path proteinTOmirnaTOprotein: 1,200,000 edges.
Path proteinTOlncrnaTOprotein: 1,200,000 edges.


In [54]:
metapath_graphs

{'proteinTOproteinTOprotein': Data(x=[18326, 150], edge_index=[2, 1200000]),
 'proteinTOmirnaTOprotein': Data(x=[18326, 150], edge_index=[2, 1200000]),
 'proteinTOlncrnaTOprotein': Data(x=[18326, 150], edge_index=[2, 1200000])}

In [55]:
from torch_geometric.nn import HANConv
import torch.nn as nn


class HANModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, metapath_graphs):
        super().__init__()

        self.edge_index_dict = {}
        for metapath_name, graph in metapath_graphs.items():
            rel = metapath_name.split('TO')[1][0] * 3  # eg: 'PMP'
            self.edge_index_dict[('protein', rel, 'protein')] = graph.edge_index

        self.metadata = (
            ['protein'],
            list(self.edge_index_dict.keys())
        )

        self.han_conv = HANConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            metadata=self.metadata,
            heads=8,
            dropout=0.6
        )
        self.lin = nn.Linear(hidden_channels, out_channels)

    def forward(self, x):
        x_dict = {'protein': x}
        out_dict = self.han_conv(x_dict, self.edge_index_dict)  # 返回 dict
        out = self.lin(out_dict['protein'])  # 取出 protein 节点的输出再过线性层
        return {"protein": out}





In [56]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for data in metapath_graphs.values():
    data.edge_index = data.edge_index.to(device)
    
train_model_with_cv(
    path="result/",
    model_name="han",
    hetero_data=hetero_data,
    model_builder=lambda: HANModel(
        in_channels=150,
        hidden_channels=88,
        out_channels=90,
        metapath_graphs=metapath_graphs
    )
)




[han] Fold 1/10

[han] Fold 2/10

[han] Fold 3/10

[han] Fold 4/10

[han] Fold 5/10

[han] Fold 6/10

[han] Fold 7/10

[han] Fold 8/10

[han] Fold 9/10

[han] Fold 10/10
han 完成训练，平均测试结果： {'accuracy': 0.5897759103641456, 'recall': 0.649494349478149, 'f1': 0.579094198819363, 'auroc': 0.7185215367848482, 'auprc': 0.5034190141131261}
